# Mô hình cuối · CNN V2 + BatchNorm

## Vì sao chọn CNN V2 + BatchNorm?

Tiêu chí chính là **macro F1 trên tập validation**. Chỉ số này lấy trung bình F1 của 7 lớp cảm xúc với trọng số bằng nhau, nên những lớp có ít ảnh vẫn được tính ngang với các lớp phổ biến.

| Mô hình trong EXP03 | Accuracy | Macro F1 |
|---|---:|---:|
| Baseline + BatchNorm | 57,68% | 53,57% |
| Deeper CNN + BatchNorm | 59,94% | 54,91% |
| **CNN V2 + BatchNorm** | 60,54% | **56,05%** |
| CNN V3 + BatchNorm | **61,44%** | 55,53% |

V2 + BatchNorm có macro F1 cao nhất trong EXP03. V3 + BatchNorm dẫn đầu về accuracy, tức tỷ lệ ảnh được dự đoán đúng trên toàn tập. Theo tiêu chí ưu tiên cân bằng giữa các lớp, nhóm chọn V2 + BatchNorm.

Checkpoint `best.pth` của V2 được lưu ở epoch 4, khi validation loss của lần huấn luyện đó thấp nhất. Sau khi chốt mô hình bằng kết quả validation, dùng cùng checkpoint để đánh giá riêng hai tập `PublicTest` và `PrivateTest` theo cột `Usage` trong `data/fer2013.csv`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader

cwd = Path.cwd()
root = next(p for p in (cwd, cwd.parent, cwd / "BTL02") if (p / "fer_data.py").exists())
sys.path.insert(0, str(root))
from fer_data import EMOTIONS, make_dataset
from models import build_model

run = root / "results/EXP03/improved_cnn_v2_bn/20261125_092340_644784"
device = torch.device("cpu")
model = build_model("improved_cnn_v2_bn").to(device)
model.load_state_dict(torch.load(run / "best.pth", map_location=device, weights_only=True))
model.eval()
print(f"Checkpoint: {run.name}/best.pth | Thiết bị: {device}")

Checkpoint: 20261125_092340_644784/best.pth | Thiết bị: cpu


## Đọc hai tập test gốc

Lấy ảnh và nhãn từ các dòng có `Usage` là `PublicTest` hoặc `PrivateTest`. Dùng cùng hàm tiền xử lý với lúc huấn luyện. Số thứ tự dòng dữ liệu trong CSV tính từ 0, bỏ qua tiêu đề.

In [2]:
splits = ("PublicTest", "PrivateTest")
chunks = pd.read_csv(
    root / "data" / "fer2013.csv",
    usecols=["Usage", "pixels", "emotion"],
    chunksize=2048,
)
test_rows = pd.concat(chunk.loc[chunk["Usage"].isin(splits)] for chunk in chunks)
test_frames = {split: test_rows.loc[test_rows["Usage"].eq(split)] for split in splits}
for split, frame in test_frames.items():
    if frame.empty:
        raise ValueError(f"fer2013.csv thiếu dữ liệu cho {split}.")
    print(f"{split}: {len(frame):,} ảnh")

PublicTest: 3,589 ảnh
PrivateTest: 3,589 ảnh


## Dự đoán và so sánh

So sánh accuracy, macro F1 và F1 từng lớp. Chênh lệch bằng kết quả PrivateTest trừ PublicTest, tính bằng điểm phần trăm. Cột `Số ảnh` trong bảng theo lớp đếm nhãn thật của từng cảm xúc.

Tệp `fer2013_test_predictions.csv` lưu tập test, vị trí ảnh trong dữ liệu gốc, nhãn thật, mã và tên lớp dự đoán, cùng xác suất softmax của lớp dự đoán.

In [3]:
class_ids = list(range(len(EMOTIONS)))
metrics = {}
by_class = pd.DataFrame(index=pd.Index(EMOTIONS, name="Cảm xúc"))
predictions = []

for split, frame in test_frames.items():
    loader = DataLoader(make_dataset(frame), batch_size=64, shuffle=False)
    predicted, confidences = [], []
    with torch.inference_mode():
        for images, _ in loader:
            scores, classes = model(images.to(device)).softmax(dim=1).max(dim=1)
            predicted.extend(classes.cpu().tolist())
            confidences.extend(scores.cpu().tolist())

    truth = frame["emotion"].to_numpy()
    scores_by_class = f1_score(truth, predicted, labels=class_ids, average=None, zero_division=0)
    metrics[split] = {
        "Số ảnh": len(frame),
        "Accuracy (%)": accuracy_score(truth, predicted) * 100,
        "Macro F1 (%)": scores_by_class.mean() * 100,
    }
    by_class[f"Số ảnh {split}"] = frame["emotion"].value_counts().reindex(class_ids, fill_value=0).to_numpy()
    by_class[f"F1 {split} (%)"] = scores_by_class * 100
    predictions.append(pd.DataFrame({
        "Usage": split,
        "fer2013_row": frame.index,
        "true_id": truth,
        "predicted_id": predicted,
        "predicted_emotion": [EMOTIONS[class_id] for class_id in predicted],
        "confidence": confidences,
    }))

summary = pd.DataFrame.from_dict(metrics, orient="index")
summary.index.name = "Tập test"
display(summary.round(2))
delta = summary.loc["PrivateTest"] - summary.loc["PublicTest"]
print(
    f"PrivateTest so với PublicTest: Accuracy {delta['Accuracy (%)']:+.2f} điểm phần trăm, "
    f"Macro F1 {delta['Macro F1 (%)']:+.2f} điểm phần trăm."
)
by_class["Chênh lệch F1 (điểm %)"] = by_class["F1 PrivateTest (%)"] - by_class["F1 PublicTest (%)"]
display(by_class.round(2))

output = run / "fer2013_test_predictions.csv"
results = pd.concat(predictions, ignore_index=True).sort_values("fer2013_row")
results.to_csv(output, index=False)
print(f"Đã lưu {len(results):,} dự đoán vào {output.relative_to(root).as_posix()}")

,Số ảnh,Accuracy (%),Macro F1 (%)
Tập test,,,
PublicTest,3589,59.01,55.91
PrivateTest,3589,61.21,56.75


PrivateTest so với PublicTest: Accuracy +2.20 điểm phần trăm, Macro F1 +0.85 điểm phần trăm.


,Số ảnh PublicTest,F1 PublicTest (%),Số ảnh PrivateTest,F1 PrivateTest (%),Chênh lệch F1 (điểm %)
Cảm xúc,,,,,
Angry,467,49.53,491,50.89,1.36
Disgust,56,48.10,55,40.00,-8.10
Fear,496,39.33,528,41.59,2.26
Happy,895,78.71,879,81.83,3.12
Sad,653,50.38,594,49.66,-0.73
Surprise,415,73.06,416,72.32,-0.74
Neutral,607,52.22,626,60.99,8.76


Đã lưu 7,178 dự đoán vào results/EXP03/improved_cnn_v2_bn/20261125_092340_644784/fer2013_test_predictions.csv


## Diễn giải kết quả

PrivateTest cao hơn PublicTest 2,20 điểm phần trăm về accuracy và 0,85 điểm về macro F1. F1 của Neutral tăng nhiều nhất (+8,76 điểm), còn Disgust giảm nhiều nhất (−8,10 điểm).

Kết quả tính trên toàn bộ hai tập gốc. Khi đối chiếu với phần dữ liệu dùng để huấn luyện checkpoint, 237 dòng PublicTest và 251 dòng PrivateTest có chuỗi pixel trùng ảnh huấn luyện. Tỷ lệ trùng này cần được tính đến khi diễn giải khả năng tổng quát hóa.